# NGIML Single Image Inference

NGIML is designed to localize image forgeries

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/juhenes/ngiml-infer.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/ngiml-infer')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run([
        'git',
        'clone',
        '--branch',
        REPO_BRANCH,
        '--single-branch',
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f'Repo ready at {REPO_DIR} on branch {REPO_BRANCH}')

In [ ]:
import json
from pathlib import Path

from google.colab import files
from src import (
    AVAILABLE_HF_CHECKPOINTS,
    DEFAULT_HF_REPO_ID,
    download_checkpoint_from_huggingface,
    plot_result,
    run_huggingface_inference,
)

CHECKPOINT_CACHE_DIR = REPO_DIR / 'checkpoints_cache'
OUTPUT_ROOT = REPO_DIR / 'outputs'

print('HF repo:', DEFAULT_HF_REPO_ID)
print('Available checkpoints:', list(AVAILABLE_HF_CHECKPOINTS))

In [ ]:
# Select in AVAILABLE_CHECKPOINTS
SELECTED_CHECKPOINT = 'casia-full.pt'

checkpoint_path = download_checkpoint_from_huggingface(
    checkpoint_name=SELECTED_CHECKPOINT,
    repo_id=DEFAULT_HF_REPO_ID,
    cache_dir=CHECKPOINT_CACHE_DIR,
)

print('Using checkpoint:', checkpoint_path)

In [ ]:

OUTPUT_DIR = None
THRESHOLD = None
NORMALIZATION_MODE = None
RESIZE_MAX_SIDE = None
CROP_SIZE = None
DEVICE = None

uploaded = files.upload()
if not uploaded:
    raise ValueError('No file uploaded.')
first_name = next(iter(uploaded.keys()))
image_path = Path('/content') / first_name

resolved_output_dir = Path(OUTPUT_DIR).expanduser().resolve() if OUTPUT_DIR else (OUTPUT_ROOT / image_path.stem)

print({
    'selected_checkpoint': SELECTED_CHECKPOINT,
    'image_path': str(image_path),
    'output_dir': str(resolved_output_dir),
    'threshold_override': THRESHOLD,
    'normalization_override': NORMALIZATION_MODE,
})

In [ ]:
result = run_huggingface_inference(
    checkpoint_name=SELECTED_CHECKPOINT,
    image_path=image_path,
    output_dir=resolved_output_dir,
    threshold=THRESHOLD,
    normalization_mode=NORMALIZATION_MODE,
    resize_max_side=RESIZE_MAX_SIDE,
    crop_size=CROP_SIZE,
    device=DEVICE,
    repo_id=DEFAULT_HF_REPO_ID,
    cache_dir=CHECKPOINT_CACHE_DIR,
)

summary = {
    'checkpoint_path': result['checkpoint_path'],
    'image_path': result['image_path'],
    'probability_mean': float(result['probability'].mean().item()),
    'probability_max': float(result['probability'].max().item()),
    'predicted_positive_ratio': float(result['binary'].mean().item()),
    'output_dir': result['output_dir'],
    'saved_paths': result['saved_paths'],
}

print(json.dumps(summary, indent=2))

In [ ]:
fig, axes = plot_result(result)